# Build a versioned yaw dataset

Drive의 검증된 yaw capture를 capture 단위로 분할해 `processed/<dataset-id>/`를 만듭니다. 개별 이미지를 무작위로 나누지 않으며 `diagnostic` 프레임은 자동 제외합니다.

## 1. 저장소와 개인 Secret 준비

Colab 왼쪽 열쇠 아이콘에 현재 저장소 읽기 권한만 있는 개인 `GITHUB_TOKEN`을 등록합니다. 토큰은 팀원과 공유하지 않습니다. 아래 설정 셀은 일회용 Colab 경로 `/content/rubber-band-gun-ai*`를 지운 뒤 최신 `main`을 다시 받으므로, 그 경로에 수동 파일을 보관하지 않습니다.

In [ ]:
import io
import shutil
import subprocess
import sys
import zipfile
from pathlib import Path

import requests
from google.colab import userdata

token = userdata.get("GITHUB_TOKEN")
if not token:
    raise RuntimeError("Colab Secret GITHUB_TOKEN is required")
headers = {"Authorization": f"Bearer {token}", "X-GitHub-Api-Version": "2022-11-28"}
commit_response = requests.get(
    "https://api.github.com/repos/kw-physical-ai/rubber-band-gun-ai/commits/main",
    headers=headers,
    timeout=60,
)
commit_response.raise_for_status()
commit_sha = commit_response.json()["sha"]
archive_response = requests.get(
    f"https://api.github.com/repos/kw-physical-ai/rubber-band-gun-ai/zipball/{commit_sha}",
    headers=headers,
    timeout=60,
)
archive_response.raise_for_status()
extract_root = Path("/content/rubber-band-gun-ai-download")
repo_root = Path("/content/rubber-band-gun-ai")
shutil.rmtree(extract_root, ignore_errors=True)
shutil.rmtree(repo_root, ignore_errors=True)
with zipfile.ZipFile(io.BytesIO(archive_response.content)) as archive:
    archive.extractall(extract_root)
shutil.move(str(next(extract_root.iterdir())), repo_root)
del token, headers, archive_response, commit_response
subprocess.check_call(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "-r",
        str(repo_root / "requirements/colab.txt"),
        "-e",
        str(repo_root),
    ]
)
sys.path.insert(0, str(repo_root / "src"))
print(f"Repository ready at {commit_sha}")

## 2. 공유 Drive 연결

팀 공유 폴더가 다른 사람의 `MyDrive` 바로 아래에 없다면, Drive 웹에서 해당 폴더의 바로가기를 `내 드라이브`에 추가하고 아래 이름과 맞춥니다.

In [ ]:
from google.colab import drive

from rubber_band_gun import ProjectWorkspace

drive.mount("/content/drive")
workspace = ProjectWorkspace(Path("/content/drive/MyDrive/rubber-band-gun-ai-data"))
workspace.create()
print(f"raw: {workspace.raw}")
print(f"processed: {workspace.processed}")

## 3. Capture 단위 split 지정

각 ID는 정확히 한 목록에만 넣습니다. `locked_test_ids`는 후보 모델과 기준을 고정하기 전까지 비워 두는 것이 기본입니다.

In [ ]:
dataset_id = "dataset-CHANGE-ME"
train_ids = []
validation_ids = []
locked_test_ids = []

if dataset_id == "dataset-CHANGE-ME":
    raise ValueError("Choose a versioned dataset_id before building")
if not train_ids and not validation_ids and not locked_test_ids:
    raise ValueError("Add at least one capture ID")
split_ids = {
    "train": train_ids,
    "validation": validation_ids,
    "locked-test": locked_test_ids,
}
split_bundles = {
    split: [workspace.raw / capture_id for capture_id in capture_ids]
    for split, capture_ids in split_ids.items()
}
for split, bundles in split_bundles.items():
    for bundle in bundles:
        if not bundle.is_dir():
            raise FileNotFoundError(f"Missing {split} capture: {bundle}")
print(split_ids)

## 4. Dataset 생성 및 재검증

동일한 dataset ID가 이미 있으면 덮어쓰지 않고 중단합니다. 수정이 필요하면 새 ID를 사용합니다.

In [ ]:
import json

from rubber_band_gun import build_yaw_dataset, verify_yaw_dataset

dataset_path = build_yaw_dataset(
    workspace.processed,
    dataset_id,
    split_bundles,
    source_commit=commit_sha,
)
report = verify_yaw_dataset(dataset_path)
print(json.dumps(report, ensure_ascii=False, indent=2))
print(f"Dataset ready: {dataset_path}")